# 01 — U-Net baseline

The generator on its own, trained directly with **MSE** (no discriminator). This is the
weakest baseline in the ablation: it learns a pixel-wise LDR→HDR mapping but has no
adversarial signal to sharpen texture. Same generator architecture as every other
notebook, via `build_generator(use_channel=False, use_spatial=False)`.

### Running on Google Colab

```python
# !git clone https://github.com/maurofalc/hdr-imaging.git
# %cd hdr-imaging
# !pip install -r requirements.txt
# from google.colab import drive; drive.mount('/content/drive')
# DATA_ROOT = "/content/drive/MyDrive/SICE"   # set this in the config cell below
```

In [ ]:
import os, sys
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# Make the shared package importable (works from the repo root or on Colab after %cd).
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
from hdr_gan import (
    build_generator, build_discriminator, load_sice,
    generator_loss, discriminator_loss, psnr, ssim,
)

In [ ]:
# --- Configuration (shared across every ablation notebook) ---
DATA_ROOT     = r"../hdr-tcc"   # folder that contains Part1/ and Part2/ (SICE)
IMG_SIZE      = 512
EPOCHS        = 50             # the thesis used 100+; keep small for quick runs
BATCH_SIZE    = 8
LEARNING_RATE = 1e-4
SEED          = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Loads SICE with the shared held-out test split (no train/test leakage).
(x_train, y_train), (x_test, y_test) = load_sice(DATA_ROOT, size=IMG_SIZE)
print("train:", x_train.shape, " test:", x_test.shape)

In [ ]:
gen = build_generator(use_channel=False, use_spatial=False,
                      input_shape=(IMG_SIZE, IMG_SIZE, 3))
gen.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
            loss=keras.losses.MeanSquaredError(),
            metrics=[psnr, ssim])
gen.summary()

In [ ]:
history = gen.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE, epochs=EPOCHS,
    validation_data=(x_test, y_test),
)

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
gen.save_weights(os.path.join("checkpoints", "01_unet.weights.h5"))
print("saved checkpoints/01_unet.weights.h5")

In [ ]:
# Reconstructions on the held-out test set.
preds = gen.predict(x_test, batch_size=BATCH_SIZE)
k = min(4, x_test.shape[0])
fig, ax = plt.subplots(k, 3, figsize=(12, 4 * k))
for i in range(k):
    for col, (img, title) in enumerate([
        (x_test[i], "LDR input"), (preds[i], "Generated HDR"), (y_test[i], "Ground truth")
    ]):
        ax[i, col].imshow(np.clip(img, 0, 1)); ax[i, col].set_title(title); ax[i, col].axis("off")
plt.tight_layout(); plt.show()

print(f"Held-out PSNR: {float(psnr(y_test, preds)):.3f}  SSIM: {float(ssim(y_test, preds)):.3f}")